# Part 3: RAG System Based on PubMed Abstracts

This notebook builds a retrieval-augmented generation system using the ChromaDB database created in Part 1 and an LLM accessed through OpenRouter.

## Imports and API Setup

This section imports the packages needed to connect to ChromaDB, load the OpenRouter API key, and call an LLM through the OpenAI-compatible OpenRouter API.

In [40]:
import os
import chromadb

from dotenv import load_dotenv
from openai import OpenAI

In [42]:
load_dotenv(".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if OPENROUTER_API_KEY is None:
    raise ValueError("OPENROUTER_API_KEY was not found. Check your environment file.")

llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "openai/gpt-oss-120b"

## ChromaDB Connection

This section connects to the persistent ChromaDB database created in Part 1. The collection contains PubMed abstracts and their metadata.

In [43]:
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "pubmed_abstracts"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print("Number of records in collection:", collection.count())

Number of records in collection: 15377


## Search Abstracts

This function performs the retrieval step. It sends the user's query to ChromaDB and returns the closest abstracts based on embedding distance.

In [44]:
def search_abstracts(query, size=5):
    response = collection.query(
        query_texts=[query],
        n_results=size
    )

    results = []

    for pmid, abstract, metadata, distance in zip(
        response["ids"][0],
        response["documents"][0],
        response["metadatas"][0],
        response["distances"][0]
    ):
        results.append({
            "pmid": pmid,
            "title": metadata["title"],
            "abstract": abstract,
            "distance": distance
        })

    return results

## Format Context

This function converts the retrieved abstracts into a text block that can be inserted into the LLM prompt. The LLM does not directly access ChromaDB; it only receives the text context created here.

In [45]:
def format_context(results):
    context_parts = []

    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"""
Source {i}
PMID: {result["pmid"]}
Title: {result["title"]}
Abstract: {result["abstract"]}
Distance: {result["distance"]}
"""
        )

    return "\n".join(context_parts)

## RAG Response Function

This function retrieves relevant abstracts and then sends those abstracts plus the question to the LLM.

In [ ]:
def rag_response(query, size=5):
    retrieved_results = search_abstracts(query, size=size)
    context = format_context(retrieved_results)

    system_message = """
You are a biomedical research assistant.
Answer the user's question using only the PubMed abstracts provided as context.
Do not use outside knowledge.
If the context does not contain enough information, say that the retrieved abstracts do not provide enough evidence.
When possible, cite PMID numbers from the context.
"""

    user_message = f"""
Question:
{query}

Retrieved PubMed abstracts:
{context}

Answer the question using only the retrieved abstracts.
"""

    completion = llm_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        temperature=0.2,
        max_tokens=400,
    )

    answer = completion.choices[0].message.content

    return {
        "query": query,
        "retrieved_results": retrieved_results,
        "answer": answer
    }

## Test Example

This cell tests the RAG system with one open-ended biomedical question.

In [49]:
query = "Why do i get headaches when i play soccer?"

rag_output = rag_response(query, size=3)

print("Question:")
print(rag_output["query"])

print("\nAnswer:")
print(rag_output["answer"])

Question:
Why do i get headaches when i play soccer?

Answer:
The abstracts you retrieved do not contain a study that directly examined headaches during soccer, but they do describe physiological mechanisms that can produce headache during intense physical activity.

* **Hyperventilation‑related headache** – In the first abstract (PMID 8947) the author reports that a stereotyped headache can be triggered by hyperventilation, which produces a burst of paroxysmal EEG activity.  During vigorous exercise such as soccer, rapid breathing and occasional hyperventilation are common, and this could provoke a similar type of headache in susceptible individuals.

* **Sympathetic (adrenergic) over‑activity** – The third abstract (PMID 26908) discusses an “autonomic theory” of migraine, proposing that an initial vasoconstriction caused by increased release of noradrenaline from sympathetic nerves leads to headache.  Physical exertion raises sympathetic tone and circulating catecholamines, which cou

### Explain with details how did you build your RAG. How does the LLM access to the database? How did you ensure that it actually used the information from the database and no pre-trained information?

I built my RAG system in three main steps. First, I used ChromaDB to retrieve the PubMed abstracts that were most semantically similar to the user's question. The abstracts had already been loaded into ChromaDB in Part 1, where each abstract was converted into an embedding. When the user enters a question, ChromaDB embeds that query and compares it to the stored abstract embeddings. It then returns the closest abstracts. Second, I formatted the retrieved abstracts into a context block. For each retrieved source, I included the PMID, title, abstract text, and distance score. Third, I sent the original question and the retrieved context to an LLM through OpenRouter using the OpenAI Python SDK.

The LLM does not directly access the database. Instead, my Python code accesses ChromaDB first, retrieves the relevant abstracts, and then passes those abstracts to the LLM inside the prompt. The LLM only receives the context that my retrieval function provides.

To encourage the LLM to use information from the database instead of relying on pre-trained knowledge, I gave it explicit instructions to answer only using the retrieved PubMed abstracts. I also instructed it to say when the retrieved abstracts do not provide enough evidence and to cite PMID numbers when possible. This does not perfectly guarantee that the model never uses outside knowledge, but it grounds the answer in the retrieved abstracts and makes the response easier to verify against the database sources.

### Question 2: How do you think you can improve your RAG system in the future? Mention at least 2 ways of improvement. Note: You don't need to implement them for the assignment.

I could store both the cleaned abstract and the original raw abstract. The cleaned version is useful for retrieval, but the raw abstract is easier for the LLM to understand because it preserves complete sentences and context. I could also split long abstracts into smaller chunks. This would allow the retrieval system to find the most relevant passages instead of retrieving entire abstracts.